# NB11 FAST — active learning trực tiếp từ preview JPG

Bản này **không chờ NB10D** và không scan lại ~113k E3 slot-images. Nó dùng luôn các `PAIRxxxx.jpg` đã có từ một run NB10A/NB10C/NB10D bất kỳ.

Mỗi preview đã chứa EVALUATION3 bên trái + Polyvore bên phải. Notebook tách 2 panel, tính các feature rẻ (pHash distance, RGB/gray/edge SSIM, Lab color delta, foreground IoU/MAE, histogram, bbox/area), rồi thử Logistic Regression / RBF-SVM / Random Forest.

Đây là **pilot nhanh** để trả lời: classical ML có học được boundary DUP/NON từ những gì mắt người thấy hay không? Nếu pilot không tốt thì dừng, không tốn giờ chạy full pipeline.


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys
import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except ImportError:
    pass

REPO_URL='https://github.com/ThinhTran2208/opisoverated.git'
BRANCH='feat/evaluation3-active-learning-nb11'
REPO_ROOT=Path('/content/opisoverated-e3-nb11-fast')
def run_git(*args,cwd=None):
    return subprocess.run(['git','-c','http.version=HTTP/1.1',*args],cwd=cwd,check=True,text=True)
if not (REPO_ROOT/'.git').is_dir():
    if REPO_ROOT.exists(): shutil.rmtree(REPO_ROOT)
    run_git('clone','--branch',BRANCH,'--single-branch',REPO_URL,str(REPO_ROOT))
else:
    run_git('fetch','origin',BRANCH,cwd=REPO_ROOT); run_git('switch',BRANCH,cwd=REPO_ROOT); run_git('pull','--ff-only','origin',BRANCH,cwd=REPO_ROOT)
subprocess.run([sys.executable,'-m','pip','install','-r',str(REPO_ROOT/'requirements-evaluation.txt')],check=True)
if str(REPO_ROOT) not in sys.path: sys.path.insert(0,str(REPO_ROOT))
from src.evaluation.evaluation3_preview_active_learning import *
from src.evaluation.evaluation3_active_learning import choose_triage_thresholds, apply_triage
from IPython.display import display
print('BRANCH:',BRANCH)


## 1. Chọn preview folder có sẵn

Notebook ưu tiên RGB run nếu có. Bạn có thể gán `PREVIEW_DIR` thủ công sang folder khác. Không cần KEY.csv.


In [ ]:
DRIVE=Path('/content/drive/MyDrive')
candidates=[
    DRIVE/'evaluation3_overlap_rgb_verifier_v0'/'rgb_aligned_trial_095_085'/'manual_review_previews',
    DRIVE/'evaluation3_overlap_phash_ssim_v2'/'phash_ssim_only'/'manual_review_previews',
    DRIVE/'evaluation3_overlap_phash_ssim_v2'/'with_exact_pixel'/'manual_review_previews',
    DRIVE/'evaluation3_overlap_ecc_verifier_v0'/'ecc_consensus_trial'/'manual_review_previews',
]
PREVIEW_DIR=next((p for p in candidates if p.is_dir() and any(p.glob('PAIR*.jpg'))),None)
if PREVIEW_DIR is None:
    raise FileNotFoundError('Không tìm thấy manual_review_previews. Gán PREVIEW_DIR thủ công tới folder PAIR*.jpg của bạn.')
all_previews=sorted(PREVIEW_DIR.glob('PAIR*.jpg'))
print('PREVIEW_DIR:',PREVIEW_DIR)
print('Preview pairs hiện có:',len(all_previews))


## 2. Build một pilot pool nhỏ

Không cần dùng hết hàng nghìn preview. Mặc định sample tối đa **600 pair** rồi cache feature CSV. Lần sau train/retrain không phải đọc ảnh lại.


In [ ]:
WORK_DIR=DRIVE/'evaluation3_active_learning_nb11_fast'
WORK_DIR.mkdir(parents=True,exist_ok=True)
FEATURE_CSV=WORK_DIR/'preview_features_600.csv'
MAX_POOL=600
RANDOM_STATE=42
if FEATURE_CSV.is_file():
    pool=pd.read_csv(FEATURE_CSV)
    print('Loaded cached features:',len(pool))
else:
    pool=build_preview_feature_pool(PREVIEW_DIR,max_pairs=MAX_POOL,random_state=RANDOM_STATE)
    pool.to_csv(FEATURE_CSV,index=False)
    print('Built + cached features:',len(pool))
display(pool.head())


## 3. Tạo seed / validation / final-test cố định

Pilot nhanh: **60 seed + 40 validation + 40 final test**. Final test không được dùng để train/chọn model. Mỗi batch còn có folder ảnh copy riêng để review dễ hơn.


In [ ]:
STATE=WORK_DIR/'state_fast.json'
SEED_SIZE,VAL_SIZE,TEST_SIZE,ROUND_SIZE=60,40,40,30
if not STATE.is_file():
    rng=np.random.default_rng(RANDOM_STATE)
    order=rng.permutation(len(pool))
    test_idx=order[:TEST_SIZE]; val_idx=order[TEST_SIZE:TEST_SIZE+VAL_SIZE]; seed_idx=order[TEST_SIZE+VAL_SIZE:TEST_SIZE+VAL_SIZE+SEED_SIZE]
    state={'test_ids':pool.iloc[test_idx].pair_id.tolist(),'val_ids':pool.iloc[val_idx].pair_id.tolist(),'seed_ids':pool.iloc[seed_idx].pair_id.tolist()}
    STATE.write_text(json.dumps(state,indent=2),encoding='utf-8')
    for name,ids in [('round_00_seed',state['seed_ids']),('validation_fixed',state['val_ids']),('FINAL_TEST_DO_NOT_USE',state['test_ids'])]:
        batch=pool[pool.pair_id.isin(ids)].copy()
        write_review_sheet(batch,WORK_DIR/f'{name}.xlsx')
        copy_batch_previews(batch,WORK_DIR/f'{name}_previews')
else:
    state=json.loads(STATE.read_text())
print({k:len(v) for k,v in state.items()})
print('Label trước: round_00_seed.xlsx + validation_fixed.xlsx')
print('Ảnh nằm trong các folder *_previews tương ứng.')


## 4. Train LR / RBF-SVM / Random Forest

Sau khi điền `DUPLICATE` / `NON_DUPLICATE` và save workbook, rerun cell này.


In [ ]:
round_files=sorted(WORK_DIR.glob('round_*.xlsx'))
labeled=attach_labels(pool,round_files)
train=labeled[labeled.target.notna() & ~labeled.pair_id.isin(state['val_ids']) & ~labeled.pair_id.isin(state['test_ids'])].copy()
val_base=pool[pool.pair_id.isin(state['val_ids'])].copy()
val=attach_labels(val_base,[WORK_DIR/'validation_fixed.xlsx'])
val=val[val.target.notna()].copy()
print('train:',len(train),train.human_label.value_counts().to_dict() if len(train) else {})
print('val:',len(val),val.human_label.value_counts().to_dict() if len(val) else {})
READY=len(train)>=20 and train.target.nunique()==2 and len(val)==len(state['val_ids']) and val.target.nunique()==2
best_model=None; best_name=None; thresholds=None
if not READY:
    print('CHƯA READY: cần >=20 train labels có cả 2 class và label đủ 40 validation.')
else:
    models,report=fit_models(train,val,RANDOM_STATE)
    display(report)
    best_name=str(report.iloc[0].model); best_model=models[best_name]
    idx=list(best_model.classes_).index(1); vp=best_model.predict_proba(xframe(val))[:,idx]
    thresholds=choose_triage_thresholds(val.target.astype(int).to_numpy(),vp,target_auto_duplicate_precision=0.98,target_auto_non_npv=0.98,minimum_auto_examples=3)
    print('BEST:',best_name)
    print('TRIAGE:',thresholds)


## 5. Active-learning round kế tiếp

Model chọn **30 pair gần p=0.5 nhất**. Bạn chỉ review các case model khó nhất, save workbook rồi quay lại cell 4.


In [ ]:
if not READY:
    print('SKIPPED')
else:
    already=set(train.pair_id)|set(state['val_ids'])|set(state['test_ids'])
    unlabeled=pool[~pool.pair_id.isin(already)].copy()
    nums=[]
    for p in WORK_DIR.glob('round_*_query.xlsx'):
        try: nums.append(int(p.stem.split('_')[1]))
        except: pass
    n=max(nums,default=0)+1
    out=WORK_DIR/f'round_{n:02d}_query.xlsx'
    if out.exists():
        print('Đã có:',out)
    else:
        q=select_uncertain(best_model,unlabeled,min(ROUND_SIZE,len(unlabeled)))
        write_review_sheet(q,out); copy_batch_previews(q,WORK_DIR/f'round_{n:02d}_query_previews')
        print('Created:',out)
        display(q[['pair_id','model_probability_duplicate','rgb_ssim','edge_ssim','mean_lab_delta','foreground_mae']].head(30))


## 6. Final test — chỉ chạy khi quyết định freeze

Khi đã đủ rounds và validation không còn cải thiện đáng kể, label `FINAL_TEST_DO_NOT_USE.xlsx`, đặt `RUN_FINAL_TEST=True`, rồi chạy đúng một lần.


In [ ]:
RUN_FINAL_TEST=False
if not RUN_FINAL_TEST:
    print('Final test đang khóa.')
elif not READY:
    print('Model chưa ready.')
else:
    test_base=pool[pool.pair_id.isin(state['test_ids'])].copy()
    test=attach_labels(test_base,[WORK_DIR/'FINAL_TEST_DO_NOT_USE.xlsx'])
    test=test[test.target.notna()].copy()
    if len(test)!=len(state['test_ids']) or test.target.nunique()!=2:
        print('Cần label đủ final test và có cả hai class.')
    else:
        # refit selected family on train + fixed validation only after model/threshold freeze
        combined=pd.concat([train,val],ignore_index=True)
        final_models,_=fit_models(combined,val,RANDOM_STATE)
        model=final_models[best_name]
        idx=list(model.classes_).index(1); p=model.predict_proba(xframe(test))[:,idx]
        print('FINAL BINARY METRICS:',binary_metrics(test.target.astype(int).to_numpy(),p))
        tri=apply_triage(p,thresholds)
        print('FINAL TRIAGE COUNTS:',pd.Series(tri).value_counts().to_dict())
